<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/01_Taller_Sentimiento_RL_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: análisis de sentimiento con regresión logística y Naive Bayes</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `03_Sentimiento_Regresion_Logistica.ipynb` y `04_Sentimiento_Naive_Bayes.ipynb` con **datos y ejercicios distintos** a los de las notas de clase. En lugar de tweets usaremos **reseñas de películas** (corpus `movie_reviews` de NLTK): textos más largos, escritos con más cuidado y **etiquetados por sus autores** según la calificación que le dieron a la película.

Enfocamos:

- el manejo de **negaciones** en el preprocesamiento,
- la regresión logística con las **características de frecuencias** del cuaderno 03 y sus limitaciones con textos largos,
- el efecto del **suavizado** en Naive Bayes y su selección por **validación cruzada**,
- la comparación honesta de modelos y el análisis de errores.

Resuelve cada ejercicio en la celda indicada. Cuando se pida una interpretación, respóndela en una celda markdown.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Exploración y división de los datos</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. Preprocesamiento con manejo de negaciones</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Regresión logística con frecuencias de palabras por clase</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Naive Bayes y la selección del suavizado</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Comparación de modelos y análisis de errores</span>](#ejercicio-5)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn scipy nltk

In [ ]:
import re
import string
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import movie_reviews

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
np.random.seed(SEED)

for recurso in ["movie_reviews", "punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Exploración y división de los datos</h2>

**Objetivo:** cargar el corpus, entender su estructura y crear una división entrenamiento/prueba sin fugas de información.

`movie_reviews` contiene 2 000 reseñas de películas en inglés, 1 000 positivas (`pos`) y 1 000 negativas (`neg`). Cada reseña es un archivo; se accede con `movie_reviews.fileids(categoria)` y `movie_reviews.raw(fileid)`.

1. Construye un `DataFrame` llamado `df` con las columnas `texto` y `etiqueta` (1 = positiva, 0 = negativa).
2. Agrega una columna `n_palabras` y reporta la media, la mediana y el máximo de palabras por reseña, por clase. ¿Hay diferencia de longitud entre clases?
3. Divide en entrenamiento (80 %) y prueba (20 %) de forma **estratificada** con `random_state=SEED`. Llama a los resultados `X_train_txt`, `X_test_txt`, `y_train`, `y_test` (las etiquetas como arreglos de NumPy).
4. Compara la longitud típica de estas reseñas con la de los tweets del cuaderno 03 (≈ 10–20 palabras). ¿Qué implicaciones crees que tiene para la representación?

In [ ]:
# Escribe tu código aquí
df = None

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Preprocesamiento con manejo de negaciones</h2>

**Objetivo:** implementar el marcado de negaciones, una técnica clásica para que los modelos de bolsa de palabras capten frases como *not good*.

Vimos que los modelos de bolsa de palabras no entienden *"not good"*. Una solución simple (Das & Chen, 2001; Pang et al., 2002) es **añadir el prefijo `NOT_` a cada palabra que sigue a una negación, hasta el siguiente signo de puntuación**:

```text
"i did n't like this movie , but the actors were great"
 -> ['i', 'did', "n't", 'NOT_like', 'NOT_this', 'NOT_movie', ',', 'but', 'the', 'actors', 'were', 'great']
```

1. Implementa `tokenizar(texto)` que pase a minúsculas y tokenice con `nltk.word_tokenize`.
2. Implementa `marcar_negaciones(tokens)` usando el conjunto `NEGACIONES` y la puntuación `PUNTUACION` dados.
3. Implementa `preprocesar(texto, negaciones=True)` que tokenice, aplique (opcionalmente) el marcado y **elimine** los signos de puntuación.
4. Pruébala con las frases de ejemplo.

In [ ]:
NEGACIONES = {"not", "no", "never", "n't", "nothing", "nobody", "nor", "neither", "without", "cannot"}
PUNTUACION = set(string.punctuation)


def tokenizar(texto):
    # Escribe tu código aquí
    pass


def marcar_negaciones(tokens):
    # Escribe tu código aquí
    pass


def preprocesar(texto, negaciones=True):
    # Escribe tu código aquí
    pass


for frase in ["I didn't like this movie, but the actors were great.",
              "Not a bad film at all. Never boring!"]:
    print(preprocesar(frase))

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Regresión logística con frecuencias de palabras por clase</h2>

**Objetivo:** aplicar a las reseñas la representación de dos características del cuaderno 03 (suma de frecuencias positivas y negativas), entrenar la regresión logística desde cero y analizar las limitaciones de esta representación con textos largos.

En el cuaderno 03 representamos cada tweet con tres números: el sesgo y las sumas de las frecuencias de sus palabras en la clase positiva y en la negativa,

$$\mathbf{x} = \left[\,1,\; \sum_{j} \text{freqs}(w_j, 1),\; \sum_{j} \text{freqs}(w_j, 0)\,\right],$$

donde $w_1, \dots, w_m$ son las palabras únicas del texto. Ahora aplicaremos la misma idea a las reseñas.

1. Implementa `construir_frecuencias(tokens_por_doc, y)`, que devuelva un `Counter` con las apariciones de cada par `(palabra, clase)`, y calcúlalo **solo** con `tok_train`.
2. Implementa `extraer_caracteristicas(tokens, freqs)` y construye las matrices `F_train` y `F_test`. Como en el cuaderno 03, aplica $\log(1 + x)$, estandariza con la media y la desviación del conjunto de **entrenamiento** y agrega la columna de sesgo. Llama a las matrices finales `X_train` y `X_test`.
3. Calcula la correlación entre las dos características y grafica las reseñas de entrenamiento en el plano (característica positiva, característica negativa). Compara con el gráfico de los tweets del cuaderno 03. ¿Por qué las dos características están tan correlacionadas? Pista: calcula también su correlación con el número de palabras únicas de la reseña.
4. Implementa `descenso_gradiente(X, y, alpha, n_iter)` como en el cuaderno 03, entrena con `alpha=0.5` y `n_iter=1000` y grafica el costo.
5. Verifica tu implementación con `LogisticRegression(C=np.inf)` de scikit-learn (sin regularización): compara los parámetros y las predicciones.
6. Reporta la exactitud en entrenamiento y en prueba y compárala con la de los tweets del cuaderno 03. ¿A qué atribuyes la diferencia entre la exactitud en entrenamiento y la de prueba? Pista: ¿las frecuencias con las que se representa una reseña de entrenamiento incluyen las palabras de esa misma reseña?

In [ ]:
# Escribe tu código aquí
def construir_frecuencias(tokens_por_doc, y):
    pass


def extraer_caracteristicas(tokens, freqs):
    pass


def sigmoide(z):
    pass


def descenso_gradiente(X, y, alpha=0.5, n_iter=1000):
    pass

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Naive Bayes y la selección del suavizado</h2>

**Objetivo:** implementar Naive Bayes multinomial con suavizado aditivo y elegir $\alpha$ con validación cruzada, sin tocar el conjunto de prueba.

1. Implementa `entrenar_nb(tokens_por_doc, y, alpha)` y `predecir_nb(modelo, tokens_por_doc)` (puedes reutilizar la lógica del cuaderno 04, pero esta vez $\alpha$ es un parámetro).
2. Verifica tu implementación: con `alpha=1` debe dar las mismas predicciones que `make_pipeline(CountVectorizer(...), MultinomialNB(alpha=1))` usando el mismo tokenizado.
3. Usa **validación cruzada estratificada de 5 particiones** (sobre el conjunto de entrenamiento) para evaluar $\alpha \in \{0.01, 0.05, 0.1, 0.5, 1, 2, 5\}$. Puedes usar `cross_val_score` con el pipeline de scikit-learn.
4. Grafica la exactitud media (con su desviación estándar) contra $\alpha$ en escala logarítmica.
5. Elige el mejor $\alpha$, reentrena con todo el conjunto de entrenamiento y evalúa **una sola vez** en prueba.
6. ¿Qué ocurre con valores de $\alpha$ muy pequeños y muy grandes? Explica por qué.

In [ ]:
# Escribe tu código aquí
def entrenar_nb(tokens_por_doc, y, alpha=1.0):
    pass


def predecir_nb(modelo, tokens_por_doc):
    pass

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Comparación de modelos y análisis de errores</h2>

**Objetivo:** comparar de forma justa los modelos y variantes de preprocesamiento, e interpretar qué aprendieron y en qué se equivocan.

1. Evalúa en prueba (F1) las cuatro combinaciones: {Naive Bayes, Regresión logística} × {con marcado de negaciones, sin marcado}. Para la regresión logística usa las características de frecuencias del ejercicio 3 (recalculando `freqs` con cada preprocesamiento); para Naive Bayes usa el `mejor_alpha` del ejercicio 4. Presenta los resultados en una tabla.
2. Con el modelo Naive Bayes con marcado de negaciones, muestra las 15 palabras con mayor razón de verosimilitud $\lambda(w)$ y las 15 con menor, primero con todo el vocabulario y luego solo con las palabras presentes en al menos 30 reseñas de entrenamiento. ¿Aparecen características `NOT_`? ¿Tienen sentido?
3. Con el mejor modelo, muestra los 3 errores en los que estaba más seguro (probabilidad más lejana a 0.5) con un fragmento del texto. Lee al menos uno y explica por qué crees que falló.
4. Conclusión: ¿qué modelo recomendarías y por qué? ¿Qué limitaciones comparten?

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Vajjala, S., Majumder, B., Gupta, A. & Surana, H. (2020). *Practical Natural Language Processing*. O'Reilly Media.
- Documentación de scikit-learn: *Working With Text Data*. [scikit-learn.org](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)
- Pang, B., Lee, L. & Vaithyanathan, S. (2002). Thumbs up? Sentiment Classification using Machine Learning Techniques. *EMNLP*.